# Полный расчётный комплект исследования
**Статья:** «Надёжность оценки моделей машинного обучения на малых несбалансированных онкологических данных»

Блокнот с нуля воспроизводит **все** результаты статьи: подготовку данных, эксперименты 1, 1б, 1в, 2, 3, 4,
анализ методологических ошибок, дополнительные метрики и рисунки. Все промежуточные и итоговые результаты
сохраняются в CSV-файлы, по которым можно независимо проверить каждую таблицу и каждый рисунок.

| Раздел | Содержание | Основные файлы |
|---|---|---|
| 1 | Загрузка и очистка данных, индексы исходных строк | `clean_*.csv`, `table1_datasets.csv`, `cervical_duplicates_by_stage.csv` |
| 2 | Эксп. 1: 200 однократных разбиений 70/30 | `exp1_single_splits.csv` |
| 3 | Эксп. 1б: CV 1×5 и 5×5 (20 повторов), полная таблица разброса | `exp1b_cv_folds.csv`, `table_sd_all.csv`, `winner_shares.csv` |
| 4 | Эксп. 1в: невложенная и вложенная CV | `exp1c_nested.csv`, `table2_nested.csv` |
| 5 | Эксп. 2: парный, скорректированный и байесовский t-критерии; чувствительность к ROPE; нормальность; A/A-проверка | `exp2_tests.csv`, `table3_*.csv`, `exp2_aa_null.csv` |
| 6 | Эксп. 3: объём выборки | `exp3_sample_size.csv`, `table_exp3_sd.csv` |
| 7 | Эксп. 4: калибровка — все 5 реализаций, бутстрэп-интервалы, полный цикл бутстрэпа (OOB) | `exp4_*.csv` |
| 8 | Контролируемый анализ ошибок на одинаковых записях и разбиениях | `pitfalls_folds.csv`, `table_pitfalls.csv` |
| 9 | Метрики с учётом дисбаланса; очистка вне/внутри CV | `extra_imbalance_metrics.csv`, `extra_preprocessing.csv` |
| 10 | Индексы всех разбиений | `splits_*.csv` |
| 11 | Рисунки 1–5 | `fig*.png` |

**Запуск:** «Среда выполнения → Выполнить все». Время выполнения ≈ 1,5–2 ч.
Если подключён Google Диск, результаты сохраняются в папку `MyDrive/ml_eval_results`; при обрыве сеанса
повторный запуск продолжит работу с того места, где она остановилась (готовые разделы пропускаются).

In [ ]:
!pip install kagglehub -q
import io, os, glob, zipfile, json, platform, warnings, itertools, time
import numpy as np, pandas as pd, requests, scipy, sklearn
from scipy import stats
from joblib import Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     GridSearchCV, cross_val_score, cross_val_predict)
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             balanced_accuracy_score, confusion_matrix)
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"   # также для параллельных процессов

SEED = 42
MODELS = ["LR", "RF", "GB", "SVM"]
DS_ORDER = ["HCC", "Cervical", "Lung"]
N_SINGLE, N_CVREP, N_NEST, N_SUB, N_CAL, B_BOOT, B_OOB = 200, 20, 5, 30, 5, 1000, 200

# Папка для результатов: Google Диск (если доступен) или локальная папка
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/ml_eval_results"
except Exception:
    OUT = "results"
os.makedirs(OUT, exist_ok=True)

def path(name): return os.path.join(OUT, name)
def save(df, name, index=False):
    df.to_csv(path(name), index=index); print("сохранено:", name, df.shape)
def done(*names): return all(os.path.exists(path(n)) for n in names)

env = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
       "scipy": scipy.__version__, "scikit-learn": sklearn.__version__, "n_cpu": os.cpu_count()}
json.dump(env, open(path("environment.json"), "w"), indent=1); print(env, "\nРезультаты:", OUT)

## 1. Загрузка и очистка данных
Очистка: (1) удаление признаков с долей пропусков > 40%; (2) удаление константных признаков;
(3) удаление идентичных записей, совпадающих по **всем признакам и значению целевой переменной**.
Шаги (1)–(2) не используют целевую переменную; шаг (3) использует её только для сравнения записей
на полное совпадение и не зависит от разбиения. Для каждой оставшейся записи сохраняется номер строки
исходного файла (`orig_row`), что позволяет в разделе 8 сравнивать варианты на одних и тех же записях.

In [ ]:
# UCI HCC Survival (165 пациентов)
z = zipfile.ZipFile(io.BytesIO(requests.get(
    "https://archive.ics.uci.edu/static/public/423/hcc+survival.zip", timeout=60).content))
name = [n for n in z.namelist() if n.endswith("hcc-data.txt") and "MACOSX" not in n][0]
hcc = pd.read_csv(z.open(name), header=None, na_values="?")
hcc.columns = ["Gender","Symptoms","Alcohol","HBsAg","HBeAg","HBcAb","HCVAb","Cirrhosis","Endemic",
 "Smoking","Diabetes","Obesity","Hemochro","AHT","CRI","HIV","NASH","Varices","Spleno","PHT","PVT",
 "Metastasis","Hallmark","Age","Grams_day","Packs_year","PS","Encephalopathy","Ascites","INR","AFP",
 "Hemoglobin","MCV","Leucocytes","Platelets","Albumin","Total_Bil","ALT","AST","GGT","ALP","TP",
 "Creatinine","Nodule","Major_Dim","Dir_Bil","Iron","Sat","Ferritin","Class"]
y_hcc = (hcc["Class"] == 0).astype(int); X_hcc = hcc.drop(columns=["Class"])

# Cervical Cancer Risk Factors (858 пациенток), целевая переменная – биопсия
z = zipfile.ZipFile(io.BytesIO(requests.get(
    "https://archive.ics.uci.edu/static/public/383/cervical+cancer+risk+factors.zip", timeout=60).content))
cerv = pd.read_csv(z.open("risk_factors_cervical_cancer.csv"), na_values="?")
TESTS = ["Hinselmann", "Schiller", "Citology"]
y_cerv = cerv["Biopsy"].astype(int)
X_cerv = cerv.drop(columns=TESTS + ["Biopsy"])

# Kaggle Survey Lung Cancer (309 записей)
import kagglehub
kpath = kagglehub.dataset_download("mysarahmadbhat/lung-cancer")
lung = pd.read_csv(glob.glob(os.path.join(kpath, "*.csv"))[0]); lung.columns = [c.strip() for c in lung.columns]
y_lung = lung["LUNG_CANCER"].map({"YES": 1, "NO": 0}).astype(int)
X_lung = lung.drop(columns=["LUNG_CANCER"]).copy()
X_lung["GENDER"] = X_lung["GENDER"].map({"M": 1, "F": 0})
for c in X_lung.columns:
    if c not in ["AGE", "GENDER"]: X_lung[c] = X_lung[c] - 1     # шкала 1/2 -> 0/1

In [ ]:
def clean(X, y, thr=0.40):
    X = X.copy()
    X = X.drop(columns=X.columns[X.isna().mean() > thr])                       # не использует y
    X = X.drop(columns=[c for c in X.columns if X[c].nunique(dropna=True) <= 1])  # не использует y
    full = X.assign(__y=y.values)
    keep = ~full.duplicated().values                    # полное совпадение признаков И метки
    orig_idx = np.where(keep)[0]
    full = full[keep]
    y2 = full.pop("__y").astype(int).reset_index(drop=True)
    return full.reset_index(drop=True), y2, orig_idx

RAW = {"HCC": (X_hcc, y_hcc), "Cervical": (X_cerv, y_cerv), "Lung": (X_lung, y_lung)}
DATA, ORIG, t1 = {}, {}, []
for ds in DS_ORDER:
    Xr, yr = RAW[ds]
    X, y, oi = clean(Xr, yr)
    DATA[ds], ORIG[ds] = (X, y), oi
    X.assign(target=y.values, orig_row=oi).to_csv(path(f"clean_{ds}.csv"), index_label="row_id")
    mino = min(y.sum(), len(y) - y.sum())
    t1.append({"dataset": ds, "n_raw": len(yr), "features_raw": Xr.shape[1],
               "minority_raw": int(min(yr.sum(), len(yr) - yr.sum())),
               "features_with_missing": int((Xr.isna().mean() > 0).sum()),
               "max_missing_%": round(float(Xr.isna().mean().max()) * 100, 1),
               "identical_rows_removed": int(len(yr) - len(y)),
               "n_clean": len(y), "features_clean": X.shape[1], "minority_clean": int(mino),
               "minority_per_feature": round(mino / X.shape[1], 2), "positive_share": round(float(y.mean()), 3)})
    print(f"{ds:9s} n={len(y):4d}  признаков={X.shape[1]:2d}  положительных={y.sum():3d} ({y.mean()*100:.1f}%)")
save(pd.DataFrame(t1), "table1_datasets.csv")

# Идентичные записи в наборе Cervical на разных этапах подготовки
st = {"raw_file_all_36_columns": int(cerv.duplicated().sum()),
      "after_excluding_3_tests": int(cerv.drop(columns=TESTS).duplicated().sum())}
save(pd.Series(st).rename("n_identical").to_frame(), "cervical_duplicates_by_stage.csv", index=True)

# Повторяющиеся записи набора Lung: число копий каждой уникальной записи
full_l = X_lung.assign(__y=y_lung.values)
gid = full_l.groupby(list(full_l.columns), dropna=False, sort=False).ngroup().values
COPIES_LUNG = np.bincount(gid)[gid[ORIG["Lung"]]] - 1          # сколько лишних копий у каждой записи
yl = DATA["Lung"][1].values
print("Lung: записей с копиями =", int((COPIES_LUNG > 0).sum()), "; лишних копий =", int(COPIES_LUNG.sum()),
      "; из них в классе 1 =", int(COPIES_LUNG[yl == 1].sum()))
print(st)

In [ ]:
def make_model(name, **kw):
    if name == "LR":  return LogisticRegression(max_iter=2000, C=1.0)
    if name == "RF":  return RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=1)
    if name == "RF_b": return RandomForestClassifier(n_estimators=200, random_state=SEED + 1, n_jobs=1)  # для A/A-проверки
    if name == "GB":  return GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=SEED)
    if name == "SVM": return SVC(kernel="rbf", probability=True, random_state=SEED)

def pipe(name, k_best=None):
    steps = [("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]
    if k_best: steps.append(("sel", SelectKBest(f_classif, k=k_best)))
    return Pipeline(steps + [("clf", make_model(name))])

def fit_auc(X, y, tr, te, m, k_best=None):
    p = pipe(m, k_best).fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
    return roc_auc_score(y.iloc[te], p)

def cv_splits(X, y, rs, n_rep=5):
    return list(RepeatedStratifiedKFold(n_splits=5, n_repeats=n_rep, random_state=rs).split(X, y))

PJ = dict(n_jobs=-1, verbose=0)

## 2. Эксперимент 1: 200 однократных разбиений 70/30 (random_state = 0…199)

In [ ]:
if not done("exp1_single_splits.csv"):
    rows = []
    for ds in DS_ORDER:
        X, y = DATA[ds]
        jobs = []
        for s in range(N_SINGLE):
            tr, te = train_test_split(np.arange(len(y)), test_size=0.30, stratify=y, random_state=s)
            jobs += [(s, m, tr, te) for m in MODELS]
        res = Parallel(**PJ)(delayed(fit_auc)(X, y, tr, te, m) for s, m, tr, te in jobs)
        rows += [{"dataset": ds, "split_seed": s, "model": m, "AUC": a} for (s, m, _, _), a in zip(jobs, res)]
        print(ds, "готово")
    save(pd.DataFrame(rows), "exp1_single_splits.csv")
exp1 = pd.read_csv(path("exp1_single_splits.csv"))
print(exp1.groupby(["dataset", "model"])["AUC"].describe(percentiles=[.025, .975]).round(3))

## 3. Эксперимент 1б: CV 1×5 и CV 5×5, 20 повторов (random_state = 1000…1019)
Для каждого повтора выполняется CV 5×5; оценка CV 1×5 – среднее по первым пяти фолдам того же повтора.
Дополнительно обучается RF с другим начальным значением генератора (RF_b, random_state = 43) – он нужен
для A/A-проверки статистических критериев в разделе 5 (две модели с одинаковым ожидаемым качеством).

In [ ]:
if not done("exp1b_cv_folds.csv"):
    rows = []
    for ds in DS_ORDER:
        X, y = DATA[ds]
        jobs = [(s, k, m, tr, te) for s in range(N_CVREP)
                for k, (tr, te) in enumerate(cv_splits(X, y, 1000 + s)) for m in MODELS + ["RF_b"]]
        res = Parallel(**PJ)(delayed(fit_auc)(X, y, tr, te, m) for s, k, m, tr, te in jobs)
        rows += [{"dataset": ds, "seed": 1000 + s, "repeat": k // 5, "fold": k % 5, "model": m, "AUC": a}
                 for (s, k, m, _, _), a in zip(jobs, res)]
        print(ds, "готово")
    save(pd.DataFrame(rows), "exp1b_cv_folds.csv")
cvf = pd.read_csv(path("exp1b_cv_folds.csv"))

In [ ]:
# Полная таблица разброса для всех моделей и наборов данных
cv5 = cvf.groupby(["dataset", "model", "seed"])["AUC"].mean()
cv1 = cvf[cvf["repeat"] == 0].groupby(["dataset", "model", "seed"])["AUC"].mean()
rows = []
for ds in DS_ORDER:
    for m in MODELS:
        a1 = exp1[(exp1.dataset == ds) & (exp1.model == m)]["AUC"]
        lo, hi = np.percentile(a1, [2.5, 97.5])
        c1, c5 = cv1.loc[(ds, m)], cv5.loc[(ds, m)]
        rows.append({"dataset": ds, "model": m,
                     "mean_single": a1.mean(), "SD_single": a1.std(ddof=1), "range95_single": hi - lo,
                     "min_single": a1.min(), "max_single": a1.max(),
                     "mean_cv1x5": c1.mean(), "SD_cv1x5": c1.std(ddof=1),
                     "mean_cv5x5": c5.mean(), "SD_cv5x5": c5.std(ddof=1),
                     "ratio_SD_single_to_cv5x5": a1.std(ddof=1) / c5.std(ddof=1),
                     "ratio_SD_cv1x5_to_cv5x5": c1.std(ddof=1) / c5.std(ddof=1)})
sd_tab = pd.DataFrame(rows)
save(sd_tab.round(4), "table_sd_all.csv")
print(sd_tab.round(3).to_string(index=False))
print("\nДиапазон отношения SD(однократное)/SD(CV 5×5):",
      round(float(sd_tab["ratio_SD_single_to_cv5x5"].min()), 1), "–", round(float(sd_tab["ratio_SD_single_to_cv5x5"].max()), 1))

# Частота, с которой каждая модель имела наибольшее значение AUC-ROC (только 4 основные модели)
def shares(df, key):
    w = df[df.model.isin(MODELS)].pivot_table(index=["dataset", key], columns="model", values="AUC")[MODELS]
    return (w.idxmax(axis=1).groupby(level=0).value_counts(normalize=True) * 100).unstack(fill_value=0)
sh = pd.concat({"single": shares(exp1, "split_seed"),
                "cv1x5": shares(cv1.reset_index(), "seed"),
                "cv5x5": shares(cv5.reset_index(), "seed")}, names=["protocol"]).reindex(columns=MODELS).fillna(0)
save(sh.round(1), "winner_shares.csv", index=True)
print("\nДоля случаев, когда модель была лучшей, %:\n", sh.round(1))

## 4. Эксперимент 1в: невложенная и вложенная кросс-валидация (внутренний цикл 2000+s, внешний 3000+s)

In [ ]:
GRIDS = {"LR":  {"clf__C": [0.01, 0.1, 1, 10]},
         "RF":  {"clf__max_depth": [None, 3, 5], "clf__min_samples_leaf": [1, 5]},
         "GB":  {"clf__learning_rate": [0.05, 0.1], "clf__max_depth": [2, 3]},
         "SVM": {"clf__C": [0.1, 1, 10], "clf__gamma": ["scale", 0.01, 0.1]}}

def nested_one(ds, m, s):
    X, y = DATA[ds]
    inner = StratifiedKFold(5, shuffle=True, random_state=2000 + s)
    outer = StratifiedKFold(5, shuffle=True, random_state=3000 + s)
    gs = GridSearchCV(pipe(m), GRIDS[m], cv=inner, scoring="roc_auc", n_jobs=1).fit(X, y)
    nested = cross_val_score(GridSearchCV(pipe(m), GRIDS[m], cv=inner, scoring="roc_auc", n_jobs=1),
                             X, y, cv=outer, scoring="roc_auc", n_jobs=1)
    return {"dataset": ds, "model": m, "rep": s, "non_nested": gs.best_score_, "nested": nested.mean(),
            "best_params_full_data": str(gs.best_params_), "grid_size": len(gs.cv_results_["params"])}

if not done("exp1c_nested.csv"):
    res = Parallel(**PJ)(delayed(nested_one)(ds, m, s) for ds in DS_ORDER for m in MODELS for s in range(N_NEST))
    nest = pd.DataFrame(res); nest["optimism"] = nest["non_nested"] - nest["nested"]
    save(nest, "exp1c_nested.csv")
nest = pd.read_csv(path("exp1c_nested.csv"))
t2 = nest.groupby(["dataset", "model"], sort=False).agg(non_nested=("non_nested", "mean"), nested=("nested", "mean"),
                                                       optimism_mean=("optimism", "mean"), optimism_sd=("optimism", "std"))
save(t2.round(4), "table2_nested.csv", index=True); print(t2.round(3))

## 5. Эксперимент 2: статистическое сравнение моделей
Для каждого из 20 повторов CV 5×5 (n = 25 фолдов): парный t-критерий; скорректированный t-критерий
(n₂/n₁ = 0,25); байесовский коррелированный t-критерий (ρ = 0,2) с ROPE = 0,005; 0,01; 0,02.
Для оценки допущения о нормальности приводится критерий Шапиро–Уилка для разностей по фолдам.

**A/A-проверка.** RF (random_state = 42) сравнивается с RF (random_state = 43): ожидаемое качество моделей
одинаково, поэтому доля «значимых» решений здесь – это эмпирическая частота ложных отклонений на реальных
данных и реальных разбиениях. Это единственная пара, для которой истинная разность известна.

In [ ]:
ROPES, RHO, RATIO = [0.005, 0.01, 0.02], 0.2, 0.25

def tests(d):
    n, dbar, var = len(d), d.mean(), d.var(ddof=1)
    out = {"dAUC": dbar, "p_paired": stats.ttest_rel(d, np.zeros(n)).pvalue if var > 0 else 1.0,
           "shapiro_p": stats.shapiro(d).pvalue if var > 0 else np.nan}
    se_c = np.sqrt((1 / n + RATIO) * var)
    out["p_corrected"] = 2 * stats.t.sf(abs(dbar) / se_c, n - 1) if var > 0 else 1.0
    sc = np.sqrt((1 / n + RHO / (1 - RHO)) * var)
    for r in ROPES:
        pa = stats.t.sf(r, n - 1, loc=dbar, scale=sc); pb = stats.t.cdf(-r, n - 1, loc=dbar, scale=sc)
        out.update({f"P_A_r{r}": pa, f"P_rope_r{r}": 1 - pa - pb, f"P_B_r{r}": pb})
    return out

pairs = list(itertools.combinations(MODELS, 2)) + [("RF", "RF_b")]
rows = []
for ds in DS_ORDER:
    w = cvf[cvf.dataset == ds].pivot_table(index=["seed", "repeat", "fold"], columns="model", values="AUC")
    for a, b in pairs:
        for seed, g in w.groupby(level=0):
            rows.append({"dataset": ds, "A": a, "B": b, "seed": seed, **tests((g[a] - g[b]).values)})
ex2 = pd.DataFrame(rows)
save(ex2, "exp2_tests.csv")

def summarize(df):
    agg = {"dAUC": ("dAUC", "mean"),
           "sig_paired_%": ("p_paired", lambda p: (p < 0.05).mean() * 100),
           "sig_corrected_%": ("p_corrected", lambda p: (p < 0.05).mean() * 100),
           "shapiro_p_lt_0.05_%": ("shapiro_p", lambda p: (p < 0.05).mean() * 100),
           "shapiro_p_median": ("shapiro_p", "median")}
    for r in ROPES:
        for k in ["A", "rope", "B"]:
            agg[f"P_{k}_r{r}"] = (f"P_{k}_r{r}", "mean")
    return df.groupby(["dataset", "A", "B"], sort=False).agg(**agg)

summ = summarize(ex2)
# Ориентируем каждую пару так, чтобы A – лучшая модель набора (по CV 5×5)
best = cv5.groupby(level=[0, 1]).mean().unstack()[MODELS].idxmax(axis=1)
oriented = []
for ds in DS_ORDER:
    bm = best[ds]
    for m in MODELS:
        if m == bm: continue
        a, b = (bm, m) if (bm, m) in pairs else (m, bm)
        r = summ.loc[(ds, a, b)].copy()
        if a != bm:          # меняем местами A и B
            r["dAUC"] = -r["dAUC"]
            for rr in ROPES:
                r[f"P_A_r{rr}"], r[f"P_B_r{rr}"] = r[f"P_B_r{rr}"], r[f"P_A_r{rr}"]
        oriented.append({"dataset": ds, "pair": f"{bm} – {m}", **r.to_dict()})
t3 = pd.DataFrame(oriented)
save(t3.round(4), "table3_best_vs_others_all_ropes.csv")
save(summ.round(4), "table3_all_pairs_all_ropes.csv", index=True)
cols = ["dataset", "pair", "dAUC", "sig_paired_%", "sig_corrected_%", "P_A_r0.01", "P_rope_r0.01", "P_B_r0.01", "shapiro_p_lt_0.05_%"]
print(t3[cols].round(3).to_string(index=False))
print("\nЧувствительность к ROPE (P лучше / P ROPE / P хуже):")
for r in ROPES:
    print(f"ROPE={r}:", t3[["dataset", "pair", f"P_A_r{r}", f"P_rope_r{r}", f"P_B_r{r}"]].round(3).to_string(index=False), sep="\n")

aa = summ.xs(("RF", "RF_b"), level=["A", "B"])
save(aa.round(4), "exp2_aa_null.csv", index=True)
print("\nA/A-проверка (RF против RF с другим random_state):\n", aa[["dAUC", "sig_paired_%", "sig_corrected_%", "P_rope_r0.01"]].round(3))

## 6. Эксперимент 3: объём выборки (25, 50, 75, 100%; 30 подвыборок; CV 2×5; random_state = 5000+r)

In [ ]:
FRACS = [0.25, 0.50, 0.75, 1.00]
def sub_one(ds, frac, r, m):
    X, y = DATA[ds]
    idx = np.arange(len(y)) if frac == 1 else \
        train_test_split(np.arange(len(y)), train_size=frac, stratify=y, random_state=5000 + r)[0]
    Xs, ys = X.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)
    aucs = [fit_auc(Xs, ys, tr, te, m) for tr, te in cv_splits(Xs, ys, 5000 + r, n_rep=2)]
    return {"dataset": ds, "fraction": frac, "sub": r, "model": m, "n": len(ys),
            "minority": int(min(ys.sum(), len(ys) - ys.sum())), "AUC": np.mean(aucs)}

if not done("exp3_sample_size.csv"):
    res = Parallel(**PJ)(delayed(sub_one)(ds, f, r, m) for ds in DS_ORDER for f in FRACS
                         for r in range(N_SUB) for m in MODELS)
    save(pd.DataFrame(res), "exp3_sample_size.csv")
ex3 = pd.read_csv(path("exp3_sample_size.csv"))
t_ex3 = ex3.groupby(["dataset", "fraction", "model"], sort=False).agg(
    n=("n", "first"), minority=("minority", "first"), AUC_mean=("AUC", "mean"), AUC_sd=("AUC", "std")).reset_index()
win3 = (ex3.pivot_table(index=["dataset", "fraction", "sub"], columns="model", values="AUC")[MODELS]
        .idxmax(axis=1).groupby(level=[0, 1]).value_counts(normalize=True).mul(100).unstack(fill_value=0))
save(t_ex3.round(4), "table_exp3_sd.csv"); save(win3.round(1), "table_exp3_winners.csv", index=True)
print(t_ex3.pivot_table(index=["dataset", "fraction"], columns="model", values="AUC_sd").round(3))

## 7. Эксперимент 4: калибровка
**7а.** Пять реализаций 5-кратной CV (random_state = 7000…7004). Для **каждой** реализации сохраняются все
прогнозы и рассчитываются показатели с 95% бутстрэп-интервалами (1000 повторов по пациентам, прогнозы фиксированы).

**7б.** Полный цикл бутстрэпа (out-of-bag): 200 бутстрэп-выборок пациентов; модель **заново обучается** на каждой
выборке и оценивается на не вошедших в неё пациентах. Такой интервал учитывает и неопределённость обучения модели,
которую бутстрэп фиксированных прогнозов не охватывает.

In [ ]:
def calib(y, p):
    p = np.clip(p, 1e-3, 1 - 1e-3)
    lg = np.log(p / (1 - p)).reshape(-1, 1)
    slope = LogisticRegression(penalty=None, max_iter=1000).fit(lg, y).coef_[0][0]
    prev = y.mean(); bs = brier_score_loss(y, p)
    return {"AUC": roc_auc_score(y, p), "BS": bs, "BSS": 1 - bs / (prev * (1 - prev)), "Slope": slope, "O_E": y.sum() / p.sum()}
MET = ["AUC", "BS", "BSS", "Slope", "O_E"]

def cal_rep(ds, m, r):
    X, y = DATA[ds]
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7000 + r)
    fold = np.empty(len(y), int)
    for k, (_, te) in enumerate(cv.split(X, y)): fold[te] = k
    p = cross_val_predict(pipe(m), X, y, cv=cv, method="predict_proba", n_jobs=1)[:, 1]
    yv = y.values; point = calib(yv, p)
    rng = np.random.default_rng(SEED + r); boot = []
    for _ in range(B_BOOT):
        i = rng.integers(0, len(yv), len(yv))
        if 0 < yv[i].sum() < len(i): boot.append([calib(yv[i], p[i])[k] for k in MET])
    lo, hi = np.percentile(np.array(boot), [2.5, 97.5], axis=0)
    row = {"dataset": ds, "model": m, "rep": r, "cv_seed": 7000 + r}
    for j, k in enumerate(MET): row.update({k: point[k], f"{k}_lo": lo[j], f"{k}_hi": hi[j]})
    pred = pd.DataFrame({"dataset": ds, "model": m, "rep": r, "row_id": np.arange(len(yv)), "fold": fold,
                         "y_true": yv, "p_pred": np.round(p, 6)})
    return row, pred

if not done("exp4_calibration_all_reps.csv", "exp4_oof_predictions_all_reps.csv"):
    res = Parallel(**PJ)(delayed(cal_rep)(ds, m, r) for ds in DS_ORDER for m in MODELS for r in range(N_CAL))
    save(pd.DataFrame([a for a, _ in res]), "exp4_calibration_all_reps.csv")
    save(pd.concat([b for _, b in res]), "exp4_oof_predictions_all_reps.csv")
cal = pd.read_csv(path("exp4_calibration_all_reps.csv"))
t4 = cal.groupby(["dataset", "model"], sort=False)[MET].agg(["mean", "min", "max"])
t4.columns = [f"{a}_{b}" for a, b in t4.columns]
save(t4.round(4), "table4_calibration_mean_min_max.csv", index=True)
print(cal.groupby(["dataset", "model"], sort=False)[MET].mean().round(3))

In [ ]:
def oob_one(ds, m, b):
    X, y = DATA[ds]; yv = y.values
    rng = np.random.default_rng(80000 + b)
    i = rng.integers(0, len(yv), len(yv)); oob = np.setdiff1d(np.arange(len(yv)), i)
    if not (0 < yv[i].sum() < len(i)) or not (0 < yv[oob].sum() < len(oob)): return None
    p = pipe(m).fit(X.iloc[i], y.iloc[i]).predict_proba(X.iloc[oob])[:, 1]
    return {"dataset": ds, "model": m, "b": b, "n_oob": len(oob), **calib(yv[oob], p)}

if not done("exp4b_oob_bootstrap.csv"):
    res = Parallel(**PJ)(delayed(oob_one)(ds, m, b) for ds in DS_ORDER for m in MODELS for b in range(B_OOB))
    save(pd.DataFrame([r for r in res if r]), "exp4b_oob_bootstrap.csv")
oobd = pd.read_csv(path("exp4b_oob_bootstrap.csv"))
q = oobd.groupby(["dataset", "model"], sort=False)[MET]
t4b = pd.concat({"median": q.median(), "lo2.5": q.quantile(0.025), "hi97.5": q.quantile(0.975)}, axis=1)
save(t4b.round(4), "table4b_oob_intervals.csv", index=True)
print(t4b.round(3))

## 8. Контролируемый анализ методологических ошибок
Корректный и ошибочный варианты оцениваются на **одних и тех же записях и одних и тех же разбиениях** CV 5×5
(random_state = 9500); тестовые части в сравниваемых вариантах полностью совпадают.
* **(а) HCC, отбор признаков:** 10 признаков по F-критерию внутри каждого обучающего фолда / по всему набору.
* **(б) Cervical, утечка:** 832 записи без результатов трёх диагностических тестов / те же 832 записи с этими результатами
  (значения тестов берутся из тех же строк исходного файла). Дополнительно – вариант без признаков Dx.
* **(в) Lung, идентичные записи:** 276 уникальных записей. Ошибочный вариант воспроизводит ситуацию, когда
  копии тестовой записи остаются в обучающей части: к обучающей части добавляются копии тестовых записей
  (по их фактическому числу в исходном файле), а для сохранения **того же объёма обучения** удаляется столько же
  случайных обучающих записей без копий. Для сравнения – вариант, где копии добавляются только к обучающим записям
  (повторы без утечки), и вариант без выравнивания объёма.

In [ ]:
SPL = {ds: cv_splits(*DATA[ds], 9500) for ds in DS_ORDER}

jobs = []   # (case, variant, dataset, X, y, fold, tr, te, model, k_best)
# (а) HCC: отбор признаков
Xh, yh = DATA["HCC"]
sel_all = SelectKBest(f_classif, k=10).fit(SimpleImputer(strategy="median").fit_transform(Xh), yh)
Xh_sel = Xh.loc[:, sel_all.get_support()]
for f, (tr, te) in enumerate(SPL["HCC"]):
    for m in MODELS:
        jobs.append(("a_feature_selection", "correct_inside_cv", Xh, yh, f, tr, te, m, 10))
        jobs.append(("a_feature_selection", "error_on_full_data", Xh_sel, yh, f, tr, te, m, None))
# (б) Cervical: те же 832 записи
Xc, yc = DATA["Cervical"]
Xc_tests = pd.concat([Xc, cerv.iloc[ORIG["Cervical"]][TESTS].reset_index(drop=True)], axis=1)
Xc_nodx = Xc.drop(columns=[c for c in Xc.columns if c.startswith("Dx")])
for f, (tr, te) in enumerate(SPL["Cervical"]):
    for m in MODELS:
        jobs.append(("b_leakage_tests", "correct_without_tests", Xc, yc, f, tr, te, m, None))
        jobs.append(("b_leakage_tests", "error_with_tests", Xc_tests, yc, f, tr, te, m, None))
        jobs.append(("b_leakage_tests", "sensitivity_without_Dx", Xc_nodx, yc, f, tr, te, m, None))
# (в) Lung: копии записей
Xl, yl = DATA["Lung"]
for f, (tr, te) in enumerate(SPL["Lung"]):
    rng = np.random.default_rng(9500 + f)
    leak = np.repeat(te, COPIES_LUNG[te])                     # копии тестовых записей в обучении
    no_copy_tr = tr[COPIES_LUNG[tr] == 0]
    drop = rng.choice(no_copy_tr, size=len(leak), replace=False)
    tr_equal = np.concatenate([np.setdiff1d(tr, drop), leak])  # тот же объём обучения
    tr_added = np.concatenate([tr, leak])
    tr_dup_train = np.concatenate([tr, np.repeat(tr, COPIES_LUNG[tr])])  # повторы только внутри обучения
    for m in MODELS:
        jobs.append(("c_duplicates", "correct_unique", Xl, yl, f, tr, te, m, None))
        jobs.append(("c_duplicates", "error_test_copies_in_train_equal_size", Xl, yl, f, tr_equal, te, m, None))
        jobs.append(("c_duplicates", "error_test_copies_in_train_added", Xl, yl, f, tr_added, te, m, None))
        jobs.append(("c_duplicates", "copies_of_train_rows_only_no_leak", Xl, yl, f, tr_dup_train, te, m, None))

if not done("pitfalls_folds.csv"):
    res = Parallel(**PJ)(delayed(fit_auc)(X, y, tr, te, m, kb) for (_, _, X, y, _, tr, te, m, kb) in jobs)
    pf = pd.DataFrame([{"case": c, "variant": v, "fold": f, "model": m, "n_train": len(tr), "n_test": len(te), "AUC": a}
                       for (c, v, X, y, f, tr, te, m, kb), a in zip(jobs, res)])
    save(pf, "pitfalls_folds.csv")
pf = pd.read_csv(path("pitfalls_folds.csv"))
base = {"a_feature_selection": "correct_inside_cv", "b_leakage_tests": "correct_without_tests", "c_duplicates": "correct_unique"}
rows = []
for (c, v, m), g in pf.groupby(["case", "variant", "model"], sort=False):
    ref = pf[(pf.case == c) & (pf.variant == base[c]) & (pf.model == m)].sort_values("fold")["AUC"].values
    d = g.sort_values("fold")["AUC"].values - ref
    rows.append({"case": c, "variant": v, "model": m, "n_train_mean": g.n_train.mean(), "AUC_mean": g.AUC.mean(),
                 "AUC_sd": g.AUC.std(), "diff_vs_correct": d.mean(),
                 "diff_lo": np.percentile(d, 2.5), "diff_hi": np.percentile(d, 97.5),
                 "folds_diff_positive_%": (d > 0).mean() * 100})
tpf = pd.DataFrame(rows); save(tpf.round(4), "table_pitfalls.csv")
print(tpf.round(3).to_string(index=False))

## 9. Дополнительно: метрики с учётом дисбаланса и очистка данных вне / внутри CV
Порог классификации на каждом фолде равен доле положительного класса в обучающей части.

In [ ]:
def fold_metrics(X, y, tr, te, m):
    p = pipe(m).fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
    yt = y.iloc[te].values; yp = (p >= y.iloc[tr].mean()).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    return {"AUC_ROC": roc_auc_score(yt, p), "PR_AUC": average_precision_score(yt, p),
            "Sensitivity": tp / (tp + fn) if tp + fn else np.nan, "Specificity": tn / (tn + fp) if tn + fp else np.nan,
            "Balanced_acc": balanced_accuracy_score(yt, yp)}

if not done("extra_imbalance_metrics.csv"):
    rows = []
    for ds in DS_ORDER:
        X, y = DATA[ds]; sp = cv_splits(X, y, 2026)
        res = Parallel(**PJ)(delayed(fold_metrics)(X, y, tr, te, m) for tr, te in sp for m in MODELS)
        rows += [{"dataset": ds, "fold": f, "model": m, **r} for (f, m), r in
                 zip([(f, m) for f in range(len(sp)) for m in MODELS], res)]
    save(pd.DataFrame(rows), "extra_imbalance_metrics.csv")
im = pd.read_csv(path("extra_imbalance_metrics.csv"))
print(im.groupby(["dataset", "model"], sort=False).agg(["mean", "std"]).drop(columns="fold").round(3))

def dedup(X, y):
    full = X.assign(__y=y.values).drop_duplicates(); y2 = full.pop("__y").astype(int)
    return full.reset_index(drop=True), y2.reset_index(drop=True)

def inside(X, y, tr, te, m, thr=0.40):
    Xtr = X.iloc[tr]
    keep = [c for c in X.columns if Xtr[c].isna().mean() <= thr and Xtr[c].nunique(dropna=True) > 1]
    return fit_auc(X[keep], y, tr, te, m), len(keep)

if not done("extra_preprocessing.csv"):
    rows = []
    for ds in DS_ORDER:
        Xd, yd = dedup(*RAW[ds])
        keep_all = [c for c in Xd.columns if Xd[c].isna().mean() <= 0.40 and Xd[c].nunique(dropna=True) > 1]
        sp = cv_splits(Xd, yd, 2027)
        a = Parallel(**PJ)(delayed(fit_auc)(Xd[keep_all], yd, tr, te, m) for tr, te in sp for m in MODELS)
        b = Parallel(**PJ)(delayed(inside)(Xd, yd, tr, te, m) for tr, te in sp for m in MODELS)
        lab = [(f, m) for f in range(len(sp)) for m in MODELS]
        rows += [{"dataset": ds, "fold": f, "model": m, "A_outside": x, "B_inside": z[0], "n_feat_B": z[1]}
                 for (f, m), x, z in zip(lab, a, b)]
    pr = pd.DataFrame(rows); pr["diff"] = pr["A_outside"] - pr["B_inside"]; save(pr, "extra_preprocessing.csv")
pr = pd.read_csv(path("extra_preprocessing.csv"))
print(pr.groupby(["dataset", "model"], sort=False)["diff"].agg(["mean", lambda s: s.abs().max()]).round(4))

## 10. Индексы всех разбиений (номер строки = `row_id` в файлах `clean_*.csv`)

In [ ]:
if not done("splits_exp1_single_70_30.csv"):
    rows = []
    for ds in DS_ORDER:
        y = DATA[ds][1]
        for s in range(N_SINGLE):
            te = train_test_split(np.arange(len(y)), test_size=0.30, stratify=y, random_state=s)[1]
            rows += [(ds, s, int(r)) for r in te]
    save(pd.DataFrame(rows, columns=["dataset", "split_seed", "test_row_id"]), "splits_exp1_single_70_30.csv")
if not done("splits_cv5x5_all.csv"):
    rows = []
    for ds in DS_ORDER:
        X, y = DATA[ds]
        for exp, seeds in [("exp1b_exp2", range(1000, 1000 + N_CVREP)), ("pitfalls", [9500]),
                           ("imbalance_metrics", [2026])]:
            for s in seeds:
                for k, (_, te) in enumerate(cv_splits(X, y, s)):
                    rows += [(exp, ds, s, k // 5, k % 5, int(r)) for r in te]
    save(pd.DataFrame(rows, columns=["experiment", "dataset", "seed", "repeat", "fold", "test_row_id"]), "splits_cv5x5_all.csv")

## 11. Рисунки 1–5

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
COL = {"LR": "#1f77b4", "RF": "#2ca02c", "GB": "#ff7f0e", "SVM": "#d62728"}
MRK = {"LR": "o", "RF": "s", "GB": "^", "SVM": "D"}
TTL = {"HCC": "Рак печени (UCI HCC)", "Cervical": "Рак шейки матки", "Lung": "Рак лёгкого"}
def letters(axes):
    for ax, l in zip(axes, ["а)", "б)", "в)"]): ax.text(-0.12, 1.06, l, transform=ax.transAxes, fontsize=12, fontweight="bold")
def finish(fig, axes, name):
    letters(axes); plt.tight_layout(); fig.savefig(path(name), dpi=600, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12, 4.4))
for ax, ds in zip(axes, DS_ORDER):
    bp = ax.boxplot([exp1[(exp1.dataset == ds) & (exp1.model == m)]["AUC"] for m in MODELS], patch_artist=True,
                    widths=.55, medianprops=dict(color="black"), flierprops=dict(marker="o", markersize=3, alpha=.5))
    for p_, m in zip(bp["boxes"], MODELS): p_.set_facecolor(COL[m]); p_.set_alpha(.85)
    ax.set_xticks(range(1, 5)); ax.set_xticklabels(MODELS); ax.axhline(.5, color="k", ls=":", lw=1)
    ax.set_title(TTL[ds]); ax.set_xlabel("Модель"); ax.grid(axis="y", alpha=.3)
axes[0].set_ylabel("AUC-ROC на тестовой выборке"); finish(fig, axes, "fig1_single_split_auc.png")

fig, axes = plt.subplots(1, 3, figsize=(12, 4.4), sharey=True)
PN = {"single": "Разбиение\n70/30", "cv1x5": "CV 1×5", "cv5x5": "CV 5×5"}
for ax, ds in zip(axes, DS_ORDER):
    bottom = np.zeros(3)
    for m in MODELS:
        v = np.array([sh.loc[(p_, ds), m] if (p_, ds) in sh.index else 0 for p_ in PN])
        ax.bar(range(3), v, bottom=bottom, color=COL[m], width=.6, label=m, edgecolor="white")
        for i, x in enumerate(v):
            if x >= 8: ax.text(i, bottom[i] + x / 2, f"{x:.0f}", ha="center", va="center", fontsize=8, fontweight="bold")
        bottom += v
    ax.set_xticks(range(3)); ax.set_xticklabels(PN.values(), fontsize=9); ax.set_title(TTL[ds]); ax.set_ylim(0, 100)
axes[0].set_ylabel("Доля случаев, когда модель\nбыла лучшей, %")
axes[2].legend(title="Модель", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False); finish(fig, axes, "fig2_winner_shares.png")

fig, axes = plt.subplots(1, 3, figsize=(12, 4.4), sharey=True)
for ax, ds in zip(axes, DS_ORDER):
    s = t_ex3[t_ex3.dataset == ds]
    for m in MODELS:
        g = s[s.model == m]; ax.plot(g.fraction, g.AUC_sd, marker=MRK[m], color=COL[m], lw=2, label=m)
    tk = s.drop_duplicates("fraction")
    ax.set_xticks(FRACS); ax.set_xticklabels([f"{int(f*100)}%\nn={n}\nредк.={k}" for f, n, k in zip(tk.fraction, tk.n, tk.minority)], fontsize=8.5)
    ax.set_title(TTL[ds]); ax.grid(alpha=.3)
axes[0].set_ylabel("Стандартное отклонение AUC-ROC"); axes[1].set_xlabel("Объём выборки (доля от исходного набора)")
axes[2].legend(title="Модель", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False); finish(fig, axes, "fig3_sample_size_sd.png")

oof = pd.read_csv(path("exp4_oof_predictions_all_reps.csv")); oof = oof[oof.rep == 0]
fig, axes = plt.subplots(1, 3, figsize=(12, 4.4))
for ax, ds in zip(axes, DS_ORDER):
    for m in MODELS:
        g = oof[(oof.dataset == ds) & (oof.model == m)]
        fp_, mp_ = calibration_curve(g.y_true, g.p_pred, n_bins=5, strategy="quantile")
        b_ = cal[(cal.dataset == ds) & (cal.model == m) & (cal.rep == 0)]["BSS"].iloc[0]
        ax.plot(mp_, fp_, marker=MRK[m], color=COL[m], lw=2, label=f"{m} (BSS={b_:.2f})")
    lim = max(ax.get_xlim()[1], ax.get_ylim()[1]); ax.plot([0, lim], [0, lim], "--", color="gray", lw=1, label="Идеальная калибровка")
    ax.set_title(TTL[ds]); ax.set_xlabel("Предсказанная вероятность"); ax.grid(alpha=.3); ax.legend(fontsize=7.5, frameon=False)
axes[0].set_ylabel("Наблюдаемая доля событий"); finish(fig, axes, "fig4_calibration.png")

VN = {"correct_inside_cv": "корректно: отбор внутри фолдов", "error_on_full_data": "ошибка: отбор на всём наборе",
      "correct_without_tests": "корректно: без трёх тестов", "error_with_tests": "ошибка: с тремя тестами",
      "sensitivity_without_Dx": "без признаков Dx",
      "correct_unique": "корректно: уникальные записи",
      "error_test_copies_in_train_equal_size": "ошибка: копии тестовых записей в обучении (тот же объём)",
      "error_test_copies_in_train_added": "ошибка: копии тестовых записей добавлены",
      "copies_of_train_rows_only_no_leak": "копии только обучающих записей (без утечки)"}
VST = [dict(marker="o", mfc="white"), dict(marker="o", mfc=None), dict(marker="s", mfc=None), dict(marker="^", mfc="white")]
fig, axes = plt.subplots(1, 3, figsize=(14, 5.6))
for ax, c in zip(axes, ["a_feature_selection", "b_leakage_tests", "c_duplicates"]):
    s = tpf[tpf.case == c]; vs = list(dict.fromkeys(s.variant))
    for j, m in enumerate(MODELS):
        for i, v in enumerate(vs):
            r = s[(s.variant == v) & (s.model == m)].iloc[0]; st_ = VST[i]
            ax.errorbar(r.AUC_mean, j + (i - (len(vs) - 1) / 2) * .18, xerr=r.AUC_sd, fmt=st_["marker"], color=COL[m],
                        mfc=st_["mfc"] or COL[m], mec=COL[m], capsize=2, ms=6)
    for i, v in enumerate(vs):
        ax.plot([], [], VST[i]["marker"], color="gray", mfc=VST[i]["mfc"] or "gray", label=VN[v])
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), fontsize=7.5, frameon=False)
    ax.set_yticks(range(4)); ax.set_yticklabels(MODELS); ax.set_xlabel("AUC-ROC (CV 5×5)"); ax.grid(axis="x", alpha=.3)
    ax.set_title({"a_feature_selection": "Отбор признаков (HCC)", "b_leakage_tests": "Утечка (рак шейки матки)",
                  "c_duplicates": "Копии записей (рак лёгкого)"}[c], fontsize=10)
finish(fig, axes, "fig5_pitfalls.png")
print("Варианты на рис. 5 (сверху вниз внутри каждой модели):", {c: list(dict.fromkeys(tpf[tpf.case == c].variant)) for c in base})

## 12. Архив всех результатов

In [ ]:
zname = "full_results.zip"
with zipfile.ZipFile(zname, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(os.listdir(OUT)):
        zf.write(path(f), f)
print("В архиве файлов:", len(os.listdir(OUT)))
try:
    from google.colab import files; files.download(zname)
except Exception:
    pass